# Đánh giá trên validation

Notebook dùng `scripts/evaluate.py` để nạp `best.pt`, dự đoán mask và xuất
metric trên tập validation. Trọng số model giữ cố định trong quá trình đánh giá.

- Kaggle: thêm code, dataset và thư mục checkpoint vào Input, bật GPU và Internet.
- Colab: đặt code, dataset trong `MyDrive/DO_AN_AI_cloud` và checkpoint trên Drive.

Chạy các cell theo thứ tự từ trên xuống. Tập test công khai chỉ có ảnh;
dùng `scripts/predict.py` để xuất dự đoán trên test.


## 1. Chọn checkpoint

E1 dùng DeepLab + CE; E2 dùng SegFormer + CE; E3 dùng kiến trúc được chọn + WCE.
`RUN_TAG` phải khớp lần huấn luyện. `CHECKPOINT_PATH` cho phép chỉ định trực tiếp
`best.pt` khi Input có nhiều checkpoint hoặc tên tag khác giá trị mặc định.
Checkpoint thiếu cấu hình nhúng cần `LEGACY_CONFIG_PATH` trỏ tới YAML tương ứng.

`LIMIT=None` đánh giá đủ 772 ảnh validation. `LIMIT=2` kiểm tra pipeline;
kết quả lưu với hậu tố `_preview` và được loại khỏi bảng so sánh cuối.


In [ ]:
MODEL = 'deeplab'
LOSS = 'ce'
RUN_TAG = 'plan20_original_loop_v1'
CHECKPOINT_PATH = None
LEGACY_CONFIG_PATH = None
LIMIT = None
MAX_FIGURES = 20


In [ ]:
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

IS_KAGGLE = Path('/kaggle/input').is_dir()
if not IS_KAGGLE and not Path('/content').is_dir():
    raise RuntimeError('Notebook này dùng trên Kaggle hoặc Google Colab.')

BASE = Path('/kaggle/working') if IS_KAGGLE else Path('/content')
DRIVE_ROOT = Path('/content/drive/MyDrive/DO_AN_AI_cloud')
if not IS_KAGGLE:
    from google.colab import drive
    drive.mount('/content/drive')

print('Nền tảng:', 'Kaggle' if IS_KAGGLE else 'Colab')
subprocess.run(['nvidia-smi'], check=True)


## 2. Chuẩn bị code và thư viện

Code được đọc từ thư mục Input hoặc giải nén từ `DO_AN_AI_cloud.zip`.
Notebook cài `requirements-cloud.txt` và dùng PyTorch CUDA có sẵn của nền tảng.
Khởi động phiên mới khi cập nhật code.


In [ ]:
PROJECT = BASE / 'DO_AN_AI'
def extract_inside(archive_path, target):
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        resolved = target.resolve()
        for member in archive.infolist():
            if not (target / member.filename).resolve().is_relative_to(resolved):
                raise ValueError('Unsafe archive path: ' + member.filename)
        archive.extractall(target)

if (PROJECT / 'scripts/train.py').is_file():
    print('Dùng source đã có:', PROJECT)
else:
    if IS_KAGGLE:
        expanded = sorted({p.parents[1] for p in Path('/kaggle/input').rglob('train.py')
                           if p.parent.name == 'scripts'
                           and (p.parents[1] / 'requirements-cloud.txt').is_file()})
        archives = sorted(Path('/kaggle/input').rglob('DO_AN_AI_cloud.zip'))
        if len(expanded) == 1:
            shutil.copytree(expanded[0], PROJECT)
        elif not expanded and len(archives) == 1:
            extract_inside(archives[0], BASE)
        else:
            raise RuntimeError('Add Input đúng một bộ code DO_AN_AI_cloud; bỏ các bộ trùng.')
    else:
        source_zip = DRIVE_ROOT / 'DO_AN_AI_cloud.zip'
        if not source_zip.is_file():
            raise FileNotFoundError('Upload code ZIP vào ' + str(source_zip))
        extract_inside(source_zip, BASE)
    assert (PROJECT / 'scripts/train.py').is_file(), 'Không thấy scripts/train.py'
print('Project:', PROJECT)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', str(PROJECT / 'requirements-cloud.txt')], check=True)
check_gpu = (
    'import torch; print(torch.__version__); '
    'assert torch.cuda.is_available(); '
    'print(torch.cuda.get_device_name(0)); '
    'print(torch.cuda.get_device_properties(0).total_memory/1024**3)'
)
subprocess.run([sys.executable, '-c', check_gpu], check=True)


## 3. Kiểm tra dataset

Dataset gồm `train/images`, `train/semantics`, `val/images` và `val/semantics`.
Kiểm tra tên file ảnh và mask khớp nhau, đủ 1.407 cặp train và 772 cặp validation.
Kaggle đọc từ Input; Colab giải nén dữ liệu vào ổ của phiên chạy.


In [ ]:
def find_dataset(search_root):
    candidates = sorted({p.parent.parent for p in search_root.rglob('semantics')
                         if p.is_dir() and p.parent.name == 'train'
                         and (p.parent.parent / 'train/images').is_dir()
                         and (p.parent.parent / 'val/images').is_dir()
                         and (p.parent.parent / 'val/semantics').is_dir()})
    if len(candidates) > 1:
        raise RuntimeError('Nhiều dataset roots; chỉ attach một bộ PhenoBench: ' + str(candidates))
    return candidates[0] if candidates else None

if IS_KAGGLE:
    DATA_ROOT = find_dataset(Path('/kaggle/input'))
    if DATA_ROOT is None:
        local_data = BASE / 'phenobench_data'
        DATA_ROOT = find_dataset(local_data) if local_data.is_dir() else None
        if DATA_ROOT is None:
            archives = sorted(Path('/kaggle/input').rglob('PhenoBench_trainval.zip'))
            if len(archives) != 1:
                raise FileNotFoundError('Add Input PhenoBench_trainval.zip hoặc train/val đã giải nén.')
            extract_inside(archives[0], local_data)
            DATA_ROOT = find_dataset(local_data)
else:
    local_data = BASE / 'phenobench_data'
    DATA_ROOT = find_dataset(local_data) if local_data.is_dir() else None
    if DATA_ROOT is None:
        extract_inside(DRIVE_ROOT / 'PhenoBench_trainval.zip', local_data)
        DATA_ROOT = find_dataset(local_data)
assert DATA_ROOT is not None, 'Không tìm được dataset root'
for split, expected in [('train', 1407), ('val', 772)]:
    images = {p.name for p in (DATA_ROOT / split / 'images').glob('*.png')}
    masks = {p.name for p in (DATA_ROOT / split / 'semantics').glob('*.png')}
    assert len(images) == expected and images == masks, (split, len(images), len(masks))
    print(split, len(images), 'image/mask pairs')
print('Data root:', DATA_ROOT)


## 4. Kiểm tra checkpoint và output

Khi `CHECKPOINT_PATH=None`, notebook tìm `best.pt` theo model, loss và tag
trong `config.yaml` cạnh checkpoint. Giữ kèm `training_log.csv` để xác nhận
đã hoàn tất ngân sách huấn luyện.

Mỗi lần đánh giá dùng thư mục output mới hoặc rỗng để tránh trộn kết quả.


In [ ]:
import yaml

if MODEL not in ('deeplab', 'segformer') or LOSS not in ('ce', 'wce'):
    raise ValueError('Kiểm tra MODEL và LOSS ở bước 1.')
EXPERIMENT_ID = 'E3' if LOSS == 'wce' else 'E1' if MODEL == 'deeplab' else 'E2'

if CHECKPOINT_PATH is None:
    search_root = Path('/kaggle/input') if IS_KAGGLE else DRIVE_ROOT / 'outputs'
    matches = []
    for path in search_root.rglob('best.pt'):
        config_path = path.parent / 'config.yaml'
        if not config_path.is_file():
            continue
        saved = yaml.safe_load(config_path.read_text(encoding='utf-8')) or {}
        run_path = saved.get('training', {}).get('output_dir', '').replace(chr(92), '/')
        same_model = saved.get('model_name') == MODEL
        same_loss = saved.get('loss', {}).get('name') == LOSS
        if same_model and same_loss and RUN_TAG in run_path.split('/') and not saved.get('smoke_test'):
            matches.append(path)
    if len(matches) != 1:
        raise RuntimeError(f'Điền CHECKPOINT_PATH ở bước 1. Các bản tìm được: {matches}')
    CHECKPOINT_PATH = matches[0]

CHECKPOINT_PATH = Path(CHECKPOINT_PATH)
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(CHECKPOINT_PATH)
run_name = EXPERIMENT_ID if LIMIT is None else EXPERIMENT_ID + '_preview'
OUTPUT = BASE / 'evaluation' / run_name
print('Experiment:', EXPERIMENT_ID)
print('Checkpoint:', CHECKPOINT_PATH)
print('Output:', OUTPUT)


## 5. Chạy evaluation

Inference đọc chế độ full/tiled từ cấu hình checkpoint. Các model cần dùng cùng
protocol để so sánh. Cấu hình `plan20` dùng ảnh nguyên và batch inference bằng 1.
`best.pt` được chọn theo validation loss thấp nhất.


In [ ]:
command = [
    sys.executable, '-u', str(PROJECT / 'scripts/evaluate.py'),
    '--checkpoint', str(CHECKPOINT_PATH), '--model-name', MODEL,
    '--data-root', str(DATA_ROOT), '--split', 'val',
    '--output-dir', str(OUTPUT), '--experiment-id', EXPERIMENT_ID,
    '--device', 'cuda', '--batch-size', '1', '--max-figures', str(MAX_FIGURES),
]
if LEGACY_CONFIG_PATH is not None:
    command.extend(['--config', str(LEGACY_CONFIG_PATH)])
if LIMIT is not None:
    command.extend(['--limit', str(LIMIT)])
subprocess.run(command, cwd=PROJECT, check=True)


## 6. Đọc kết quả

`metrics.json` lưu tỉ lệ từ 0 đến 1; cell này hiển thị phần trăm.
`n/a` biểu thị metric chưa xác định do mẫu số bằng 0.
Weed Coverage bằng Weed Recall khi spray mask là vùng dự đoán Weed.
Estimated Pixel-Area Reduction đo phần diện tích pixel không phun;
lượng thuốc thực tế cần được đo riêng.


In [ ]:
import json

metrics = json.loads((OUTPUT / 'metrics.json').read_text(encoding='utf-8'))
print('Số ảnh đã đánh giá:', metrics['dataset']['evaluated_images'])
print('Quality flags:', metrics['quality_flags'])
scores = {
    'mIoU': metrics['segmentation']['miou'],
    'Crop IoU': metrics['segmentation']['crop_iou'],
    'Weed IoU': metrics['segmentation']['weed_iou'],
    'Weed Recall / Coverage': metrics['segmentation']['weed_recall'],
    'Crop Wrongly Sprayed Rate': metrics['spraying']['crop_wrongly_sprayed_rate'],
    'Spray Area Ratio': metrics['spraying']['spray_area_ratio'],
    'Estimated Pixel-Area Reduction': metrics['spraying']['estimated_pixel_area_reduction'],
}
for label, value in scores.items():
    print(f'{label}: {value:.2%}' if value is not None else f'{label}: n/a')


## 7. Lưu kết quả

ZIP gồm `metrics.json`, confusion matrix, metric từng ảnh, predictions,
spray masks, error maps và hình minh họa.
Kaggle lưu ZIP trong `/kaggle/working`, tải từ bảng Output.
Colab sao chép ZIP vào Google Drive.

Đánh giá từng thí nghiệm, sau đó dùng
[result_analysis.ipynb](result_analysis.ipynb) để tổng hợp kết quả.


In [ ]:
archive_path = BASE / f'evaluation_{OUTPUT.name}.zip'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
    for path in sorted(OUTPUT.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(OUTPUT.parent).as_posix())

if IS_KAGGLE:
    print('ZIP:', archive_path)
    print('Tải trong bảng Output bên phải: mở /kaggle/working, tìm', archive_path.name)
    print('Chọn dấu ba chấm cạnh file, chọn Download.')
else:
    shutil.copy2(archive_path, DRIVE_ROOT / archive_path.name)
    print('Đã lưu:', DRIVE_ROOT / archive_path.name)
